# Imports

In [ ]:
from cytools import Polytope, Cone

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, diagnostics, lattice

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import numpy as np

In [ ]:
import flint

## Hard-coded Manwe's CY

In [ ]:
if True:
    # Manwe
    verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
    heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]
elif False:
    # h11=10
    verts   = [[1, 0, 0, 0], [0, 1, 0, 0], [-14, -9, -3, -1], [-3, -2, -1, 1], [0, 0, 0, 1], [0, 0, 1, 0], [-8, -5, -2, 0], [-4, -3, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 20.49999999999999, 0.0, -2.4999999999999964, 4.249999999999997, 6.749999999999995, 0.0, -7.999999999999995, -3.2499999999999982, -4.499999999999998, 7.249999999999999, -2.749999999999999, -5.249999999999997, 0.0]
elif True:
    # h11=16
    verts   = [[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [-32, -21, -9, -1], [-10, -7, -3, 1], [-3, -2, -1, 1], [-1, -1, 0, 1]]
    heights = [0.0, 0.0, 3.500000000000006, 0.0, 60.750000000000014, 51.750000000000014, -17.749999999999996, 0.0, -18.25, 16.500000000000014, -20.25, -20.750000000000004, 10.000000000000009, 4.5000000000000036, 24.250000000000007, 0.0, -0.24999999999999506, -3.5000000000000036, 36.0, -5.000000000000006, -3.500000000000005]
else:
    # h11=20
    verts = [[1, 0, 0, 0], [0, 1, 0, 0], [-3, -2, -2, 2], [-24, -16, -6, -1], [-12, -8, -6, 3], [-9, -6, -5, 3], [-5, -3, -3, 2], [0, 0, 0, 1], [0, 0, 1, 0]]


In [ ]:
p       = Polytope(verts)
#t       = p.triangulate(heights=heights)
#cy      = t.cy()

## Set the conifold-related info

In [ ]:
# get the conifold charge
# -----------------------
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

In [ ]:
t  = conis[0].dual_triangulation()
cy = t.cy()

In [ ]:
data = cydata.CYData.from_cy(cy, coni_curve=q)

In [ ]:
Qmax = data.h11+data.h21+4

# Study fancier methods

In [ ]:
min_N_pts = 10000

grading = np.sum(data.H_cob, axis=0)
if data.h11<20:
    p  = Zp.mindeg_pvec_gurobi(data)
    mindeg = np.dot(p,grading)

    ps = Zp.pvecs(data, min_deg=mindeg, deg_window=max(5,mindeg//100), min_N_pts=min_N_pts)
else:
    totskc = np.rint(Cone(hyperplanes=data.H_cob).tip_of_stretched_cone()).astype(int)
    print(np.dot(totskc, grading))

    ps = Zp.pvecs(data, max_deg = 237_000)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
verbosity = 0

good_pfvs = []
for d in tqdm(range(1,200+1)):
for p in tqdm(ps):
    mat, Z, Binter = Zp.coniMellipsoid(p, data)
    proj = np.hstack([ np.zeros((data.h11-1,1), dtype=int), np.eye(data.h11-1, dtype=int) ])

    for d in tqdm(range(1,200+1)):
        # get a basis of the lattice Kperp % d = 0
        # ----------------------------------------
        A = proj@Z@Binter
        A_fl = flint.fmpz_mat(A.tolist())
        
        A_extended    = np.hstack([A, -d*np.eye(A.shape[0], dtype=int) ])
        A_extended_fl = flint.fmpz_mat(A_extended.tolist())
    
        Ht, Tt = A_extended_fl.transpose().hnf(transform=True)
        H = Ht.transpose()
        T = Tt.transpose()
        
        # get the null space
        first_null = 0
        for j in range(H.ncols()):
            for i in range(H.nrows()):
                if H[i,j] != 0:
                    break
            else:
                first_null = j
                break
        
        null_fl = flint.fmpz_mat(T.nrows()//2, H.ncols()-first_null)
        for i in range(null_fl.nrows()):
            for j in range(null_fl.ncols()):
                null_fl[i,j] = T[i,j+first_null]
        null = np.array(null_fl.transpose().lll().transpose().tolist()).astype(int)
    
        assert np.all((A@null % d) == 0)
    
        # sort null to maximize leading 0s in (Binter@null)[0]
        sort_inds = np.argsort((Binter@null)[0]!=0)
        null = null[:,sort_inds]
        
        # write/solve the ellipsoid problem
        # ---------------------------------
        L = np.linalg.cholesky(null.T@mat@null)
        vs = lattice.fp_iterative_lincut(
            L, d*Qmax,
            linvec = (Binter@null)[0], linmin = 13, linmax=float('inf'),
            max_N_out=10_000_000, eps=1e-4)[0]
        if verbosity >= 1:
            print(len(vs),end=',')
        if len(vs) == 10_000_000:
            print('satd')
        #print(len(vs),end=',')
    
        # convert all good lattice vectors to c coefficients
        cs = null@np.array(vs).T
        Ms = Binter@cs
        mask = Ms[0] > 12
        cs = cs[:,mask]
        Ms = Ms[:,mask]
        
        if not any(mask):
            continue
    
        # compute the Ms, Ks
        Ks = Z@Ms
        assert np.allclose(Ks[1:]/d,Ks[1:]//d)
        Ks = Ks//d
    
        # check that K0 can be set...
        Qperp = -np.sum(Ks[1:]*Ms[1:], axis=0)
        divisible = ((Qperp-Qmax) % Ms[0]) == 0
        Qperp = Qperp[divisible]
        Ms = Ms[:,divisible]
        Ks = Ks[:,divisible]
    
        if verbosity >= 1:
            print(f"{Ms.shape[1]} passed tadpole cut...")
    
        # check that N has nonzero det N, check it has nonzero det
        for Q,K,M in zip(Qperp,Ks.T,Ms.T):
            N = (data.kappa_cob@M)[1:,1:]
            if np.linalg.matrix_rank(N)==N.shape[0]:
                pass
            else:
                continue
    
            # set K0
            K[0] = (Q-Qmax)//M[0]
    
            #print(f"K = {K.tolist()}")
            #print(f"M = {M.tolist()}")
            #print(f"p = {p.tolist()}")
            #print()
            pfv = diagnostics.PFV(data, K=K, M=M, silent=False)
            if not pfv.check_all():
                #print(pfv.Kprime)
                pass
            else:
                print(":)")
                good_pfvs.append(pfv)

In [ ]:
KMs = []
for pfv in good_pfvs:
    KMs.append(pfv.K.tolist()+pfv.M.tolist())
KMs = np.array(KMs)

In [ ]:
print(len(KMs), len(np.unique(KMs,axis=0)))

In [ ]:
M0s = [pfv.M[0] for pfv in good_pfvs]

In [ ]:
import matplotlib.pyplot as plt
plt.hist(M0s)

In [ ]:
((data.kappa_cob@M)[1:,1:]).shape

In [ ]:
np.linalg.matrix_rank((data.kappa_cob@M)[1:,1:])

In [ ]:
sort_inds = np.argsort((Binter@null)[0]!=0)

In [ ]:
(Binter@null[:,sort_inds])

In [ ]:
Binter[:,sort_inds]

In [ ]:
Ms[0]

In [ ]:
(Qperp-Qmax) % Ms[0]

In [ ]:
Binter.shape

In [ ]:
np.abs(np.linalg.det((data.kappa_cob@M)[1:,1:])) >= 0.5

In [ ]:
K = [-16, 0, -3, 3, -1, 0, 2, -7, -1, 18, 3, -3, 0, -7, -20, -8]
M = [14, 128, 40, 10, 84, 30, 70, 60, 57, 42, 37, 36, 28, 26, 14, -4]

pfv = diagnostics.PFV(data, K=K, M=M)

In [ ]:
pfv

In [ ]:
K//d

In [ ]:
d = 8
p = [-45, -21, -9, -34, -16, -33, -31, -24, -28, -10, -24, -13, -18, -10, 15]
c = [-18, -16, -24, -8, -4, 12, 8, -28, 20, 4, -4, -16, -52, 80, -52]

In [ ]:
mat, Z, Binter = Zp.coniMellipsoid(p, data)

In [ ]:
M = Binter@c
Knaive = Z@Binter@c

In [ ]:
-np.dot(M,Knaive/d)/Qmax

In [ ]:
-np.dot(M,Knaive/d)

In [ ]:
Knaive

In [ ]:
M

In [ ]:
Qmax

In [ ]:
np.linalg.eig(null.T@mat@null)